<a href="https://colab.research.google.com/github/rachana1707-S/Adaptive-MedRAG/blob/feature%2Fneeraj-generation/notebooks/03_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03: LLM Generation and Evaluation (Neeraj)

Part of our CS 6120 project, Adaptive RAG for Healthcare. This notebook covers my part: answering PubMedQA questions with an LLM, first without retrieval (no-RAG baseline) and later with retrieved evidence (RAG), then evaluating accuracy, evidence support, and response time.

## Part 1: PubMedQA Loading and Exploration

In [1]:
# install the hugging face datasets library so we can download PubMedQA
!pip install -q datasets

In [2]:
from datasets import load_dataset

# pqa_labeled is the expert-labeled subset (1000 questions), the only one we can use for accuracy
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
print(pubmedqa)

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


In [3]:
# look at the first example to see what each field holds
labeled_data = pubmedqa["train"]
first_example = labeled_data[0]

print("pubid:", first_example["pubid"])
print("question:", first_example["question"])
print("final_decision:", first_example["final_decision"])
print()
print("context keys:", first_example["context"].keys())
print()
print("long_answer:", first_example["long_answer"])

pubid: 21645374
question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
final_decision: yes

context keys: dict_keys(['contexts', 'labels', 'meshes', 'reasoning_required_pred', 'reasoning_free_pred'])

long_answer: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.


In [4]:
# the context field has the abstract split into sections (background, methods, results...)
abstract_sections = first_example["context"]["contexts"]
section_labels = first_example["context"]["labels"]

# print each section with its label
for section_index in range(len(abstract_sections)):
    print(f"[{section_labels[section_index]}]")
    print(abstract_sections[section_index])
    print()

[BACKGROUND]
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.

[RESULTS]
The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye M

**Note on leakage:** `long_answer` is the conclusion of the abstract and basically gives away the answer. We never use it in prompts or in the retrieval corpus. We only use `question` (input), `context.contexts` (evidence for RAG), and `final_decision` (gold label).

In [5]:
from collections import Counter

# count how many yes/no/maybe gold answers there are
label_counts = Counter(labeled_data["final_decision"])

total_examples = len(labeled_data)
for label, count in label_counts.items():
    percentage = (count / total_examples) * 100
    print(f"{label}: {count} ({percentage:.1f}%)")

yes: 552 (55.2%)
no: 338 (33.8%)
maybe: 110 (11.0%)


The labels are imbalanced: 55.2% yes, 33.8% no, 11.0% maybe. So a model that always says "yes" already gets 55.2% accuracy, which is the bar to beat. Because "maybe" is rare, we also need per-class metrics and macro F1, not just accuracy.

In [6]:
# check roughly how long each abstract is, in words
abstract_word_counts = []

for example in labeled_data:
    # glue the abstract sections together, then count the words
    abstract_sections = example["context"]["contexts"]
    full_abstract = " ".join(abstract_sections)

    words = full_abstract.split()
    word_count = len(words)
    abstract_word_counts.append(word_count)

shortest = min(abstract_word_counts)
longest = max(abstract_word_counts)
average = round(sum(abstract_word_counts) / len(abstract_word_counts), 1)

print("shortest abstract:", shortest, "words")
print("longest abstract:", longest, "words")
print("average abstract:", average, "words")

shortest abstract: 49 words
longest abstract: 398 words
average abstract: 200.2 words


Abstracts are 49 to 398 words (about 200 on average). At roughly 1.5 tokens per word, even 5 full abstracts are only about 3,000 tokens, well within Qwen2.5's context window. So we don't need to cut abstracts short to fit them in the prompt.

## Part 2: LLM Setup

In [7]:
import torch

# make sure colab gave us a GPU
gpu_is_available = torch.cuda.is_available()
print("GPU is available:", gpu_is_available)
print("GPU name:", torch.cuda.get_device_name(0))

GPU is available: True
GPU name: Tesla T4


In [8]:
# transformers runs the hugging face model, accelerate puts it on the GPU
!pip install -q -U transformers accelerate

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# compared 1.5B and 3B on the dev set and picked 1.5B (see Part 3 notes)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# the tokenizer converts text to token ids and back
tokenizer = AutoTokenizer.from_pretrained(model_name)

# load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
)

print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0


In [10]:
# generation settings for no-RAG and RAG
MAX_NEW_TOKENS = 20
SYSTEM_MESSAGE = "You are a helpful biomedical research assistant."

def ask_model(user_message):
    # chat models expect a list of messages, each with a role
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_message},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # turn the prompt into token ids and move them to the GPU
    model_inputs = tokenizer(prompt_text, return_tensors="pt")
    input_ids = model_inputs["input_ids"].to(model.device)
    attention_mask = model_inputs["attention_mask"].to(model.device)

    output_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )

    # the output starts with the prompt tokens, so skip those and keep only the answer
    prompt_length = input_ids.shape[1]
    all_tokens = output_ids[0]
    answer_tokens = all_tokens[prompt_length:]

    # turn the answer tokens back into text
    answer_text = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    return answer_text

In [11]:
import time

# quick test on the first 3 questions to check that everything works
for example_index in range(3):
    example = labeled_data[example_index]
    question = example["question"]

    user_message = question + "\nAnswer with one word: yes, no, or maybe."

    start_time = time.time()
    model_answer = ask_model(user_message)
    end_time = time.time()

    response_time_ms = round((end_time - start_time) * 1000)

    print("Question:", question)
    print("Model answer:", model_answer)
    print("Gold answer:", example["final_decision"])
    print("Time:", response_time_ms, "ms")
    print("-" * 50)

Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Model answer: Yes
Gold answer: yes
Time: 1972 ms
--------------------------------------------------
Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Model answer: Yes
Gold answer: no
Time: 138 ms
--------------------------------------------------
Question: Syncope during bathing in infants, a pediatric form of water-induced urticaria?
Model answer: Yes
Gold answer: yes
Time: 192 ms
--------------------------------------------------


Two things from this test: the model capitalizes its answer ("Yes" vs gold "yes"), so we need a function to clean up the answer before comparing. And the first question is much slower than the rest because the GPU warms up, so we run one throwaway question before timing anything.

## Part 3: No-RAG Baseline

We split the 1000 questions into a **dev set (100)** for checking prompts and code, and a **test set (900)** that is only used for the final numbers. This way we never tune the prompt on the questions we report results on. The RAG runs will use the same 900 test questions.

In [12]:
import random

# list of positions 0 to 999, one per question
all_indexes = list(range(len(labeled_data)))

# shuffle with a fixed seed so the split is the same every time
random.seed(42)
random.shuffle(all_indexes)

# first 100 shuffled positions go to dev, the other 900 go to test
dev_indexes = all_indexes[:100]
test_indexes = all_indexes[100:]

print("dev set size:", len(dev_indexes))
print("test set size:", len(test_indexes))

dev set size: 100
test set size: 900


In [13]:
import json
import os

# save the pubmed ids (not positions) so teammates can use the exact same split
dev_question_ids = []
for index in dev_indexes:
    dev_question_ids.append(str(labeled_data[index]["pubid"]))

test_question_ids = []
for index in test_indexes:
    test_question_ids.append(str(labeled_data[index]["pubid"]))

split_info = {
    "seed": 42,
    "dev_question_ids": dev_question_ids,
    "test_question_ids": test_question_ids,
}

# write it to data/pubmedqa_split.json
os.makedirs("data", exist_ok=True)
with open("data/pubmedqa_split.json", "w") as split_file:
    json.dump(split_info, split_file, indent=2)

print("Saved split to data/pubmedqa_split.json")

Saved split to data/pubmedqa_split.json


In [14]:
# what each label means in PubMedQA
LABEL_DEFINITIONS = (
    "Use these meanings for the labels:\n"
    "yes = the research results support the idea in the question\n"
    "no = the research results do not support it, or show the opposite\n"
    "maybe = the research results are mixed or inconclusive"
)

In [15]:
def build_no_rag_prompt(question):
    # no-RAG: the model only sees the question, no abstract or evidence
    instructions = "Answer the following biomedical research question."
    answer_format = "Reply with exactly one word: yes, no, or maybe."

    prompt = instructions + "\n\n" + LABEL_DEFINITIONS + "\n\n" + "Question: " + question + "\n\n" + answer_format
    return prompt

In [16]:
def extract_label(answer_text):

    cleaned_text = answer_text.lower()

    punctuation_marks = [".", ",", "!", "?", ":", ";", "\"", "'", "*"]
    for mark in punctuation_marks:
        cleaned_text = cleaned_text.replace(mark, " ")

    words = cleaned_text.split()

    for word in words:
        if word == "yes":
            return "yes"
        if word == "no":
            return "no"
        if word == "maybe":
            return "maybe"

    return "unknown"

In [17]:
# check the extractor on some made-up answers
sample_answers = [
    "Yes",
    "no.",
    "Maybe, the evidence is unclear",
    "The answer is: No",
    "I am not sure",
]

for sample_answer in sample_answers:
    print(sample_answer, "->", extract_label(sample_answer))

Yes -> yes
no. -> no
Maybe, the evidence is unclear -> maybe
The answer is: No -> no
I am not sure -> unknown


In [18]:
import time


def run_no_rag(indexes):
    results = []

    for position in range(len(indexes)):
        index = indexes[position]
        example = labeled_data[index]
        question = example["question"]

        prompt = build_no_rag_prompt(question)
        start_time = time.time()
        raw_answer = ask_model(prompt)
        end_time = time.time()

        response_time_ms = round((end_time - start_time) * 1000)
        predicted_label = extract_label(raw_answer)

        # one result in our team's shared format
        result = {
            "question_id": str(example["pubid"]),
            "question": question,
            "raw_answer": raw_answer,
            "prediction": predicted_label,
            "gold_answer": example["final_decision"],
            "response_time_ms": response_time_ms,
        }
        results.append(result)

        # print progress every 50 questions
        if (position + 1) % 50 == 0:
            print("Finished", position + 1, "of", len(indexes))

    return results

In [19]:
# warm-up question
warm_up_answer = ask_model("Is water wet? Reply with one word: yes or no.")
print("Warm-up done:", warm_up_answer)

Warm-up done: Yes.


In [20]:
# run the no-RAG baseline on the dev set
dev_results = run_no_rag(dev_indexes)

# count how often each label was predicted
yes_count = 0
no_count = 0
maybe_count = 0
unknown_count = 0

for result in dev_results:
    if result["prediction"] == "yes":
        yes_count = yes_count + 1
    elif result["prediction"] == "no":
        no_count = no_count + 1
    elif result["prediction"] == "maybe":
        maybe_count = maybe_count + 1
    else:
        unknown_count = unknown_count + 1

print("Predicted yes:", yes_count)
print("Predicted no:", no_count)
print("Predicted maybe:", maybe_count)
print("Unknown:", unknown_count)

# look at the first 5 raw answers to see what the model actually wrote
print()
for result in dev_results[:5]:
    print(result["raw_answer"], "->", result["prediction"], "| gold:", result["gold_answer"])

Finished 50 of 100
Finished 100 of 100
Predicted yes: 15
Predicted no: 7
Predicted maybe: 78
Unknown: 0

No -> no | gold: yes
Yes -> yes | gold: no
Maybe -> maybe | gold: no
Maybe -> maybe | gold: maybe
Yes -> yes | gold: yes


### Prompt versions tried on the dev set

We first ran these experiments with Qwen2.5-3B:

- **v1** (question + "reply with yes, no, or maybe"): no-RAG predicted maybe 96/100 times.
- **v2** (also asked the model to commit to yes/no when it can): maybe 99/100, so it didn't help. We dropped it.
- **v3** (v1 + the PubMedQA label definitions): no-RAG still 0 yes / 2 no / 98 maybe.

Three versions gave the same behavior, so we stopped tuning the no-RAG prompt. Without evidence, 3B just refuses to guess. v3 is the final prompt, used for every experiment.

In [21]:
def build_gold_abstract_prompt(question, abstract_sections):
    # only the contexts field, never long_answer
    abstract_text = " ".join(abstract_sections)

    instructions = "Answer the following biomedical research question based on the abstract below."
    answer_format = "Reply with exactly one word: yes, no, or maybe."

    prompt = instructions + "\n\n" + LABEL_DEFINITIONS + "\n\n" + "Abstract: " + abstract_text + "\n\n" + "Question: " + question + "\n\n" + answer_format
    return prompt

# oracle check: give the model the real abstract for each dev question
yes_count = 0
no_count = 0
maybe_count = 0
unknown_count = 0
correct_count = 0

for index in dev_indexes:
    example = labeled_data[index]
    prompt = build_gold_abstract_prompt(example["question"], example["context"]["contexts"])

    raw_answer = ask_model(prompt)
    predicted_label = extract_label(raw_answer)

    if predicted_label == "yes":
        yes_count = yes_count + 1
    elif predicted_label == "no":
        no_count = no_count + 1
    elif predicted_label == "maybe":
        maybe_count = maybe_count + 1
    else:
        unknown_count = unknown_count + 1

    if predicted_label == example["final_decision"]:
        correct_count = correct_count + 1

print("Predicted yes:", yes_count)
print("Predicted no:", no_count)
print("Predicted maybe:", maybe_count)
print("Unknown:", unknown_count)
print("Correct:", correct_count, "out of", len(dev_indexes))

Predicted yes: 86
Predicted no: 14
Predicted maybe: 0
Unknown: 0
Correct: 61 out of 100


In [22]:
# how well would always answering "yes" do on the dev set?
dev_gold_yes_count = 0

for index in dev_indexes:
    if labeled_data[index]["final_decision"] == "yes":
        dev_gold_yes_count = dev_gold_yes_count + 1

print("Always-yes on dev:", dev_gold_yes_count, "out of", len(dev_indexes))

Always-yes on dev: 53 out of 100


### Choosing the model (dev set, prompt v3)

| | Qwen2.5-3B | Qwen2.5-1.5B |
|---|---|---|
| No-RAG (yes / no / maybe) | 0 / 2 / 98 | 15 / 7 / 78 |
| Gold abstract correct | 50 / 100 | 61 / 100 |
| Gold abstract (yes / no / maybe) | 30 / 22 / 48 | 86 / 14 / 0 |

Always-yes on dev gets 53/100. Before adding the label definitions, 3B with the gold abstract only got 34/100.

We picked **Qwen2.5-1.5B-Instruct**. Our rule was to choose the model that does better with the gold abstract, and 1.5B is the only one that beats always-yes (61 vs 53). It also downloads faster on Colab. Weaknesses to keep in mind: with evidence it never predicts "maybe" and leans heavily toward "yes".

The outputs above are from the 1.5B run.

### Final no-RAG run on the test set

Same model, same prompt (v3), run once on the 900 test questions.

In [23]:
# the real baseline run on the 900 test questions
test_results = run_no_rag(test_indexes)

# save the results as json
os.makedirs("results", exist_ok=True)
results_path = "results/no_rag_qwen2.5-1.5b_test.json"

with open(results_path, "w") as results_file:
    json.dump(test_results, results_file, indent=2)

print("Saved", len(test_results), "results to", results_path)

Finished 50 of 900
Finished 100 of 900
Finished 150 of 900
Finished 200 of 900
Finished 250 of 900
Finished 300 of 900
Finished 350 of 900
Finished 400 of 900
Finished 450 of 900
Finished 500 of 900
Finished 550 of 900
Finished 600 of 900
Finished 650 of 900
Finished 700 of 900
Finished 750 of 900
Finished 800 of 900
Finished 850 of 900
Finished 900 of 900
Saved 900 results to results/no_rag_qwen2.5-1.5b_test.json


In [24]:
from google.colab import files

files.download("results/no_rag_qwen2.5-1.5b_test.json")
files.download("data/pubmedqa_split.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>